In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "bronze-rp")
dbutils.widgets.text("catalogo", "catalog_RP")
dbutils.widgets.text("esquema", "bronze")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

ruta = f"abfss://{container}@adlssmartdata1201rp.dfs.core.windows.net/Automobile.csv"

In [0]:
automobile_schema = StructType(
    fields=[
        StructField("make", StringType(), False),
        StructField("fuel-type", StringType(), True),
        StructField("aspiration", StringType(), True),
        StructField("num-of-doors", StringType(), True),
        StructField("body-style", StringType(), True),
        StructField("price", DoubleType(), True),
    ]
)

In [0]:
df_automobile_final = spark.read\
.option('header', True)\
.schema(automobile_schema)\
.csv(ruta)

In [0]:
automobile_selected_df = df_automobile_final.select(
    col("make"), col("fuel-type"), col("body-style"), col("price")
)

In [0]:
automobile_renamed_df = (
    automobile_selected_df.withColumnRenamed("make", "Marca")
    .withColumnRenamed("fuel-type", "TipoConbustible")
    .withColumnRenamed("body-style", "TipoVehiculo")
    .withColumnRenamed("price", "Precio")
)

In [0]:
automobile_final_df = automobile_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
automobile_final_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.automobile")